In [1]:
import os
os.chdir("D:\jupyter_code\CNBT-ACPred-main\CNBT-ACPred-main")

In [6]:
import csv
from collections import OrderedDict

def parse_fasta(fasta_file):
    """Read FASTA file and return a dict {id: sequence}."""
    seq_dict = {}
    current_id = None
    current_seq = []
    with open(fasta_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if current_id:
                    seq_dict[current_id] = ''.join(current_seq)
                # extract ID (e.g., "MMCD_0001")
                current_id = line[1:].split()[0]   # take first word after '>'
                current_seq = []
            else:
                current_seq.append(line)
        if current_id:
            seq_dict[current_id] = ''.join(current_seq)
    return seq_dict

def read_predictions_csv(pred_file):
    """Read mmcd_dual_sturct_10000_result.csv and return dict {peptide: row_data}."""
    pred_dict = {}
    with open(pred_file, newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            peptide = row['Peptide Sequence']
            pred_dict[peptide] = row
    return pred_dict

def main():
    fasta_file = "mmcd_non_struct_10000.fasta"
    pred_file = "mmcd_non_struct_10000_result.csv"
    toxin_file = "non_struct_toxinpred.csv"
    output_file = "merged_non_struct_predictions.csv"

    # Step 1: map FASTA IDs to sequences
    id_to_seq = parse_fasta(fasta_file)
    print(f"Loaded {len(id_to_seq)} sequences from FASTA.")

    # Step 2: read the prediction CSV (keyed by peptide)
    pred_by_seq = read_predictions_csv(pred_file)
    print(f"Loaded {len(pred_by_seq)} predictions.")

    # Step 3: read toxin CSV and merge
    merged_rows = []
    with open(toxin_file, newline='') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames + list(pred_by_seq[next(iter(pred_by_seq))].keys())
        # use OrderedDict to keep column order
        for row in reader:
            subject = row['Subject']
            seq = id_to_seq.get(subject)
            if seq is None:
                print(f"Warning: {subject} not found in FASTA, skipping.")
                continue
            pred_row = pred_by_seq.get(seq)
            if pred_row is None:
                print(f"Warning: sequence for {subject} not found in prediction CSV, skipping.")
                continue
            # merge dictionaries
            merged = {**row, **pred_row}
            merged_rows.append(merged)

    # Step 4: write merged CSV
    if merged_rows:
        with open(output_file, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=merged_rows[0].keys())
            writer.writeheader()
            writer.writerows(merged_rows)
        print(f"Merged {len(merged_rows)} records into {output_file}")
    else:
        print("No records merged.")

if __name__ == "__main__":
    main()

Loaded 10000 sequences from FASTA.
Loaded 10000 predictions.
Merged 10000 records into merged_non_struct_predictions.csv


In [8]:
import pandas as pd

# 读取CSV文件
df = pd.read_csv('merged_non_struct_predictions.csv')

# 根据Prediction和Prediction Class进行分类统计
toxin_acp = ((df['Prediction'] == 'Toxin') & (df['Prediction Class'] == 1)).sum()
toxin_non_acp = ((df['Prediction'] == 'Toxin') & (df['Prediction Class'] == 0)).sum()
nontoxin_acp = ((df['Prediction'] == 'Non-Toxin') & (df['Prediction Class'] == 1)).sum()
nontoxin_non_acp = ((df['Prediction'] == 'Non-Toxin') & (df['Prediction Class'] == 0)).sum()

print("统计结果：")
print(f"toxinACP: {toxin_acp}")
print(f"toxinnonACP: {toxin_non_acp}")
print(f"nontoxinACP: {nontoxin_acp}")
print(f"nontoxinnonACP: {nontoxin_non_acp}")

统计结果：
toxinACP: 2491
toxinnonACP: 45
nontoxinACP: 7076
nontoxinnonACP: 388
